## NSR On $\texttt{Nations}$

### 数据集准备

In [1]:
import re
import time
import random
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
from pykeen.datasets import Nations

@dataclass
class RuleLearnConfig:
    symmetry_threshold: float = 0.5
    inverse_threshold: float = 0.5
    min_support: int = 8
    min_conf: float = 0.6
    composition_mode: str = "relaxed"

# Load PyKEEN Nations dataset
def load_pykeen_nations():
    dataset = Nations()
    
    # entity/relation mapping
    entity_to_id = dataset.training.mapped_triples[:, 0].numpy() # just an array
    
    # a better way is to use pykeen's built-in mapping
    entity_to_id = dataset.entity_to_id
    relation_to_id = dataset.relation_to_id
    id_to_entity = {v: k for k, v in entity_to_id.items()}
    id_to_relation = {v: k for k, v in relation_to_id.items()}
    
    def to_triples(triples_factory):
        triples = triples_factory.mapped_triples.tolist()
        return [(h, r, t) for h, r, t in triples]
    
    train_triples = to_triples(dataset.training)
    valid_triples = to_triples(dataset.validation)
    test_triples = to_triples(dataset.testing)
    
    all_triples = train_triples + valid_triples + test_triples
    
    return train_triples, valid_triples, test_triples, all_triples, entity_to_id, relation_to_id, id_to_entity, id_to_relation

train_triples, valid_triples, test_triples, all_triples, entity_to_id, relation_to_id, id_to_entity, id_to_relation = load_pykeen_nations()

num_entities = len(entity_to_id)
num_relations = len(relation_to_id)

print(f"Entities: {num_entities}")
print(f"Relations: {num_relations}")
print(f"Train triples: {len(train_triples)}")
print(f"Valid triples: {len(valid_triples)}")
print(f"Test triples: {len(test_triples)}")


Entities: 14
Relations: 55
Train triples: 1592
Valid triples: 199
Test triples: 201


/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def evaluate_ranking(score_fn, eval_triples, num_entities):
    ranks = []
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_10 = 0
    
    for h, r, t in eval_triples:
        scores_dict = score_fn(h, r)
        # Sort entities by score, descending
        sorted_entities = sorted([e for e in range(num_entities)], 
                                 key=lambda e: scores_dict.get(e, 0.0), 
                                 reverse=True)
        
        # In case of ties, rank could vary. We will use the first occurrence index + 1
        # It's better to break ties randomly or properly. Let's use simple index.
        try:
            rank = sorted_entities.index(t) + 1
        except ValueError:
            rank = num_entities
            
        ranks.append(rank)
        if rank <= 1: hits_at_1 += 1
        if rank <= 3: hits_at_3 += 1
        if rank <= 10: hits_at_10 += 1
        
    n = len(eval_triples)
    mr = sum(ranks) / n
    mrr = sum(1.0 / rank for rank in ranks) / n
    hits_at_1 = (hits_at_1 / n) * 100
    hits_at_3 = (hits_at_3 / n) * 100
    hits_at_10 = (hits_at_10 / n) * 100
    
    return {"MR": mr, "MRR": mrr, "Hits@1": hits_at_1, "Hits@3": hits_at_3, "Hits@10": hits_at_10}

def evaluate_ranking_filtered(score_fn, eval_triples, num_entities, known_triples):
    known_hr2t = defaultdict(set)
    for h, r, t in known_triples:
        known_hr2t[(h, r)].add(t)
        
    ranks = []
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_10 = 0
    
    for h, r, t in eval_triples:
        scores_dict = score_fn(h, r)
        
        # Add small random noise to break ties
        entity_scores = []
        for e in range(num_entities):
            score = scores_dict.get(e, 0.0)
            if e in known_hr2t[(h, r)] and e != t:
                score = -float('inf')
            entity_scores.append((e, score + random.uniform(0, 1e-6)))
            
        sorted_entities = sorted(entity_scores, key=lambda x: x[1], reverse=True)
        sorted_e = [x[0] for x in sorted_entities]
        
        try:
            rank = sorted_e.index(t) + 1
        except ValueError:
            rank = num_entities
            
        ranks.append(rank)
        if rank <= 1: hits_at_1 += 1
        if rank <= 3: hits_at_3 += 1
        if rank <= 10: hits_at_10 += 1
        
    n = len(eval_triples)
    mr = sum(ranks) / n
    mrr = sum(1.0 / rank for rank in ranks) / n
    hits_at_1 = (hits_at_1 / n) * 100
    hits_at_3 = (hits_at_3 / n) * 100
    hits_at_10 = (hits_at_10 / n) * 100
    
    return {"MR": mr, "MRR": mrr, "Hits@1": hits_at_1, "Hits@3": hits_at_3, "Hits@10": hits_at_10}

def print_metrics(name, met):
    print(f"{name}: MR: {met['MR']:.4f}, MRR: {met['MRR']:.4f}, Hits@1: {met['Hits@1']:.2f}%, Hits@3: {met['Hits@3']:.2f}%, Hits@10: {met['Hits@10']:.2f}%")


### Model Definition

In [ ]:
def discover_sym_inv(train_triples, num_relations, sym_min_support=2, sym_min_conf=0.1, inv_min_support=2, inv_min_conf=0.1):
    """
    发现对称关系和逆关系，并考虑support和 confidence的置信度。
    """
    pairs_by_r = defaultdict(set)
    pair_to_relations = defaultdict(set)
    
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))
        pair_to_relations[(h, t)].add(r) 

    symmetric_relations = set()
    for r in range(num_relations):
        pairs = pairs_by_r[r]
        if not pairs: continue
        
        # 统计在这个关系下，有多少对的逆方向也存在于同一个关系下
        sym_count = sum(1 for (h, t) in pairs if (t, h) in pairs)
        conf = sym_count / len(pairs)
        if sym_count >= sym_min_support and conf >= sym_min_conf:
            symmetric_relations.add(r)

    inverse_pairs = set()
    for r1 in range(num_relations):
        p1 = pairs_by_r[r1]
        if not p1: continue
        for r2 in range(num_relations):
            if r1 == r2: continue
            p2 = pairs_by_r[r2]
            if not p2: continue
            
            # 统计在 r1 的边里，有多少条的逆方向存在于 r2
            inv_count = sum(1 for (h, t) in p1 if (t, h) in p2)
            conf = inv_count / len(p1)
            
            if inv_count >= inv_min_support and conf >= inv_min_conf:
                inverse_pairs.add((r1, r2))
                
    return pair_to_relations, symmetric_relations, inverse_pairs

def discover_multi_topology(train_triples, num_relations, min_support=8, min_conf=0.1):
    """
    独立挖掘三种拓扑规则：
    1. Chain: r_i(h, x) & r_j(x, t) => r_k(h, t)
    2. V-Struct (汇聚): r_i(h, x) & r_j(t, x) => r_k(h, t)
    3. Fork (发散): r_i(x, h) & r_j(x, t) => r_k(h, t)
    """
    pairs_by_r = defaultdict(set)
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))
        
    rules_chain, rules_vstruct, rules_fork = {}, {}, {}
    
    for r_i in range(num_relations):
        pi = pairs_by_r[r_i]
        if not pi: continue
        
        h_to_x = defaultdict(list)
        x_to_h = defaultdict(list)
        for h, x in pi:
            h_to_x[h].append(x)
            x_to_h[x].append(h)
            
        for r_j in range(num_relations):
            pj = pairs_by_r[r_j]
            if not pj: continue
            
            x_to_t = defaultdict(list)
            t_to_x = defaultdict(list)
            for x, t in pj:
                x_to_t[x].append(t)
                t_to_x[t].append(x)
                
            chain_pairs, vstruct_pairs, fork_pairs = set(), set(), set()
            
            # 1. Chain: h->x from pi, x->t from pj
            for h, xs in h_to_x.items():
                for x in xs:
                    for t in x_to_t[x]:
                        chain_pairs.add((h, t))
                        
            # 2. V-Struct: h->x from pi, t->x from pj
            for x, hs in x_to_h.items():
                if x in t_to_x:
                    for h in hs:
                        for t in t_to_x[x]:
                            vstruct_pairs.add((h, t))
                            
            # 3. Fork: x->h from pi, x->t from pj
            for x, hs in h_to_x.items():
                if x in x_to_t:
                    for h in hs:
                        for t in x_to_t[x]:
                            fork_pairs.add((h, t))
                            
            def eval_rule(pairs, rule_dict):
                sup = len(pairs)
                if sup < min_support: return
                best_rk, best_hit = None, -1
                for r_k in range(num_relations):
                    hit = len(pairs & pairs_by_r[r_k])
                    if hit > best_hit:
                        best_rk, best_hit = r_k, hit
                conf = best_hit / sup
                if conf >= min_conf:
                    rule_dict[(r_i, r_j)] = best_rk
                    
            eval_rule(chain_pairs, rules_chain)
            eval_rule(vstruct_pairs, rules_vstruct)
            eval_rule(fork_pairs, rules_fork)
            
    return rules_chain, rules_vstruct, rules_fork

class NSR1990FullTopology:
    def __init__(self, num_entities, num_relations):
        self.N, self.M = num_entities, num_relations
        self.W_beta_beta = np.zeros((self.N * self.M, self.N * self.M))
        self.W_beta_gamma = np.zeros((self.N, self.M))
        
        self._tails_by_hr = defaultdict(list)
        self._heads_by_tr = defaultdict(list)
        
        self.symmetric_relations = set()
        self.inverse_map = {}
        
        self.rules_chain = defaultdict(list)
        self.rules_vstruct = defaultdict(list)
        self.rules_fork = defaultdict(list)

    def _vec_index(self, e, r): return r * self.N + e

    def fit(self, train_triples):
        self.train_triples = list(train_triples)
        hr_freq = defaultdict(int)
        for h, r, t in train_triples:
            self.W_beta_beta[self._vec_index(t, r), self._vec_index(h, r)] = 1.0
            self._tails_by_hr[(h, r)].append(t)
            self._heads_by_tr[(t, r)].append(h)
            hr_freq[(h, r)] += 1
        
        for (h, r), freq in hr_freq.items():
            self.W_beta_gamma[h, r] = 1.0 + np.log1p(freq)
        return self

    def set_rules(self, sym, inv, chain_r, vstruct_r, fork_r):
        self.symmetric_relations = set(sym)
        self.inverse_map = {r1: r2 for r1, r2 in inv}
        for r1, r2 in inv: self.inverse_map[r2] = r1
        
        for (r_i, r_j), r_k in chain_r.items(): self.rules_chain[r_k].append((r_i, r_j))
        for (r_i, r_j), r_k in vstruct_r.items(): self.rules_vstruct[r_k].append((r_i, r_j))
        for (r_i, r_j), r_k in fork_r.items(): self.rules_fork[r_k].append((r_i, r_j))

    def _compositionality_infer(self, h_q, r_q):
        scores_comp = {}
        topologies_used = []

        # 1. Chain: h_q --ri--> x --rj--> t
        for r_i, r_j in self.rules_chain.get(r_q, []):
            hit_any = False
            for x in self._tails_by_hr.get((h_q, r_i), []):
                amp_hx = self.W_beta_gamma[h_q, r_i]
                for t in self._tails_by_hr.get((x, r_j), []):
                    amp_xt = self.W_beta_gamma[x, r_j]
                    scores_comp[t] = scores_comp.get(t, 0.0) + (amp_hx * amp_xt)
                    hit_any = True
            if hit_any: topologies_used.append(f"chain({r_i},{r_j})")

        # 2. V-Struct: h_q --ri--> x <--rj-- t
        for r_i, r_j in self.rules_vstruct.get(r_q, []):
            hit_any = False
            for x in self._tails_by_hr.get((h_q, r_i), []):
                amp_hx = self.W_beta_gamma[h_q, r_i]
                for t in self._heads_by_tr.get((x, r_j), []):
                    amp_tx = self.W_beta_gamma[t, r_j]
                    scores_comp[t] = scores_comp.get(t, 0.0) + (amp_hx * amp_tx)
                    hit_any = True
            if hit_any: topologies_used.append(f"vstruct({r_i},{r_j})")

        # 3. Fork: h_q <--ri-- x --rj--> t
        for r_i, r_j in self.rules_fork.get(r_q, []):
            hit_any = False
            for x in self._heads_by_tr.get((h_q, r_i), []):
                amp_xh = self.W_beta_gamma[x, r_i]
                for t in self._tails_by_hr.get((x, r_j), []):
                    amp_xt = self.W_beta_gamma[x, r_j]
                    scores_comp[t] = scores_comp.get(t, 0.0) + (amp_xh * amp_xt)
                    hit_any = True
            if hit_any: topologies_used.append(f"fork({r_i},{r_j})")
            
        return scores_comp, topologies_used

    def infer(self, h_q, r_q):
        if r_q in self.symmetric_relations:
            res = self._score_via_rel(h_q, r_q)
            if res: return res
        
        scores_inv = {}
        if r_q in self.inverse_map:
            scores_inv = self._score_via_rel(h_q, self.inverse_map[r_q])
        
        scores_comp, _ = self._compositionality_infer(h_q, r_q)
        
        if scores_inv or scores_comp:
            merged = {}
            for e in set(scores_inv) | set(scores_comp):
                merged[e] = scores_inv.get(e, 0.0) + scores_comp.get(e, 0.0)
            return merged

        return self._base_rr_infer(h_q, r_q)

    def _score_via_rel(self, h_q, r_used):
        heads = self._heads_by_tr.get((h_q, r_used), [])
        if not heads: return {}
        return {x: float(self.W_beta_gamma[x, r_used]) for x in heads}

    def _base_rr_infer(self, h_q, r_q):
        col = self.W_beta_beta[:, self._vec_index(h_q, r_q)]
        scores_arr = col.reshape(self.N, self.M, order="F").sum(axis=1)
        return {int(e): float(s) for e, s in enumerate(scores_arr) if s > 0}

    def learn_all_rules(self, min_support=8, min_conf=0.1, sym_min_support=2, sym_min_conf=0.1, inv_min_support=2, inv_min_conf=0.1):
        pair_rels, sym, inv = discover_sym_inv(self.train_triples, self.M, sym_min_support, sym_min_conf, inv_min_support, inv_min_conf)
        c, v, f = discover_multi_topology(self.train_triples, self.M, min_support, min_conf)
        self.set_rules(sym, inv, c, v, f)
        
        all_rules = {}
        for k, val in c.items(): all_rules[f"chain_{k}"] = val
        for k, val in v.items(): all_rules[f"vstruct_{k}"] = val
        for k, val in f.items(): all_rules[f"fork_{k}"] = val
        return all_rules, sym, inv


In [ ]:
import tracemalloc

out_strs = []

out_strs.append("=" * 100)
out_strs.append("Evaluating Enhanced NSR (NSR1990FullTopology) on PyKEEN NATIONS")
out_strs.append("=" * 100)

tracemalloc.start()
# 1. 构建与装载模型
model_enhanced = NSR1990FullTopology(num_entities, num_relations)
model_enhanced.fit(train_triples)

# 2. 规则学习
out_strs.append("Learning enhanced rules (Chain, V-Struct, Fork capable)...")
start_time = time.perf_counter()
# Adjust min_support / min_conf for Nations which is quite small and sparse. Maybe min_support=2, min_conf=0.1
enhanced_comp_rules, sym_rels, inv_rels = model_enhanced.learn_all_rules(min_support=5, min_conf=0.3)
learn_time = time.perf_counter() - start_time
out_strs.append(f"Learned rules count: {len(enhanced_comp_rules)} (Done in {learn_time:.2f}s)")

# 3. 打印对称关系和逆关系
out_strs.append(f"\n--- Symmetric Relations (Total: {len(sym_rels)}) ---")
for r in list(sym_rels)[:5]:
    out_strs.append(f"- {id_to_relation[r]}")

out_strs.append(f"\n--- Inverse Relations (Total: {len(inv_rels)}) ---")
for r1, r2 in list(inv_rels)[:5]:
    out_strs.append(f"- {id_to_relation[r1]}  <-->  {id_to_relation[r2]}")

out_strs.append("\n--- Compositional Rules Sample (Top 5) ---")
c = 0
for rule_type_r1_r2, r_k in enhanced_comp_rules.items():
    if c >= 5: break
    rtype = rule_type_r1_r2.split("_")[0]
    r_i_j_str = rule_type_r1_r2.replace(f"{rtype}_", "").strip("()")
    r_i, r_j = eval(f"({r_i_j_str})")
    
    r_i_name = id_to_relation[r_i]
    r_j_name = id_to_relation[r_j]
    r_k_name = id_to_relation[r_k]
    out_strs.append(f"[{rtype.upper()}] {r_i_name} & {r_j_name} => {r_k_name}")
    c += 1

# 4. 评测推理性能
out_strs.append("\n--- Summary Metrics (Enhanced NSR) ---")
start = time.perf_counter()
enhanced_raw = evaluate_ranking(lambda h, r: model_enhanced.infer(h, r), test_triples, num_entities)
enhanced_raw_t = time.perf_counter() - start

start = time.perf_counter()
enhanced_filtered = evaluate_ranking_filtered(lambda h, r: model_enhanced.infer(h, r), test_triples, num_entities, all_triples)
enhanced_filt_t = time.perf_counter() - start

out_strs.append(f"Enhanced NSR - Test (raw)  [time={enhanced_raw_t:.2f}s]: MR={enhanced_raw['MR']:.4f}, MRR={enhanced_raw['MRR']:.4f}, Hits@1={enhanced_raw['Hits@1']:.2f}%")
out_strs.append(f"Enhanced NSR - Test (filtered)  [time={enhanced_filt_t:.2f}s]: MR={enhanced_filtered['MR']:.4f}, MRR={enhanced_filtered['MRR']:.4f}, Hits@1={enhanced_filtered['Hits@1']:.2f}%")

res_text = "\n".join(out_strs)
print(res_text)


In [ ]:
# ==================================================================================
# 额外：对 min_support 和 min_conf 进行超参数调优 (Grid Search) 
# 加入了针对 对称(sym) 和可逆(inv) 规则单独设定的 support/conf 搜索
# ==================================================================================
print("\n" + "=" * 100)
print("Hyperparameter Tuning for NSR: Topology, Symmetric, and Inverse Rules")
print("=" * 100)

# 组合拓扑的超参数
comp_support_list = [2, 4, 6]
comp_conf_list = [0.1, 0.3, 0.5]

# 对称和逆关系由于Nations数据集很小，我们可以给它们不同的或更低的要求
# 为了避免网格爆炸，我们将 sym 和 inv 的超参数绑定成一组 `[sym_inv_sup, sym_inv_conf]` 一起搜索
sym_inv_support_list = [1, 2] 
sym_inv_conf_list = [0.1, 0.3]

results_grid = []

best_mrr = -1.0
best_params = {}

for c_sup in comp_support_list:
    for c_conf in comp_conf_list:
        for si_sup in sym_inv_support_list:
            for si_conf in sym_inv_conf_list:
                
                # 重新挖掘并设置规则
                rules, sym, inv = model_enhanced.learn_all_rules(
                    min_support=c_sup, min_conf=c_conf,
                    sym_min_support=si_sup, sym_min_conf=si_conf,
                    inv_min_support=si_sup, inv_min_conf=si_conf
                )
                
                # 在验证集上执行过滤后的评估 (避免知识泄漏)
                val_metrics = evaluate_ranking_filtered(
                    lambda h, r: model_enhanced.infer(h, r), 
                    valid_triples, 
                    num_entities, 
                    all_triples
                )
                
                mrr = val_metrics["MRR"]
                hits1 = val_metrics["Hits@1"]
                
                results_grid.append({
                    "min_support": c_sup,
                    "min_conf": c_conf,
                    "sym_inv_support": si_sup,
                    "sym_inv_conf": si_conf,
                    "Num Tot Rules": len(rules) + len(sym) + len(inv),
                    "Val MRR": mrr,
                    "Val Hits@1": hits1
                })
                
                print(f"min_sup={c_sup}, min_conf={c_conf:<3}, si_sup={si_sup}, si_conf={si_conf:<3} | " 
                      f"Tot Rules: {len(rules)+len(sym)+len(inv):<4} | Val MRR={mrr:.4f} | Val Hits@1={hits1:.2f}%")
                
                if mrr > best_mrr:
                    best_mrr = mrr
                    best_params = {
                        "min_support": c_sup, 
                        "min_conf": c_conf,
                        "sym_min_support": si_sup,
                        "sym_min_conf": si_conf,
                        "inv_min_support": si_sup,
                        "inv_min_conf": si_conf
                    }

# 转换为 DataFrame 并打印最佳结果排序
df_grid = pd.DataFrame(results_grid)
print("\n--- Grid Search Results (Sorted by Val MRR) ---")
print(df_grid.sort_values(by="Val MRR", ascending=False).head(15).to_string(index=False)) # 只展示前 15

print("\n" + "=" * 100)
print(f"Best Parameters on Validation: {best_params}")
print("=" * 100)

# 重新用最佳参数进行规则学习，并在测试集上做最终评估
print(f"\nRe-evaluating explicitly on Test Set using BEST parameters...")
final_rules, final_sym, final_inv = model_enhanced.learn_all_rules(**best_params)

best_test_metrics = evaluate_ranking_filtered(
    lambda h, r: model_enhanced.infer(h, r), 
    test_triples, 
    num_entities, 
    all_triples
)

# 覆盖前面给后续 KGE baseline 比较用的全局变量 enhanced_filtered 
enhanced_filtered = best_test_metrics

print_metrics("Best Enhanced NSR - Test (filtered)", best_test_metrics)

## Other Models On $\texttt{NATONS}$

In [3]:
import time
import torch
import numpy as np
import pandas as pd
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from pykeen.evaluation import RankBasedEvaluator
from pykeen.models import TransE, DistMult, ComplEx, RotatE, ConvE, RESCAL

device_nations = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 100)
print("NATIONS: 其他 KGE 模型测试 (使用带有早期停止和针对性超参调优验证)")
print("=" * 100)
print(f"Device: {device_nations}")

def build_tf_nations(triples, create_inverse=False):
    return TriplesFactory(
        mapped_triples=np.asarray(triples, dtype=np.int64),
        entity_to_id=entity_to_id,
        relation_to_id=relation_to_id,
        create_inverse_triples=create_inverse,
    )

def metric_dict(metric_results):
    return {
        "MR": float(metric_results.get_metric("mean_rank")),
        "MRR": float(metric_results.get_metric("mean_reciprocal_rank")),
        "Hits@1": float(metric_results.get_metric("hits_at_1")) * 100,
        "Hits@3": float(metric_results.get_metric("hits_at_3")) * 100,
        "Hits@10": float(metric_results.get_metric("hits_at_10")) * 100,
    }

# 用于最后 evaluation 的 base (没有 Inverse Triples)
train_eval_tf = build_tf_nations(train_triples, create_inverse=False)
valid_eval_tf = build_tf_nations(valid_triples, create_inverse=False)
test_eval_tf = build_tf_nations(test_triples, create_inverse=False)

def evaluate_split_nations(model, split_tf, filtered=False):
    evaluator = RankBasedEvaluator(filtered=filtered)
    kwargs = {"batch_size": 128}
    if filtered:
        kwargs["additional_filter_triples"] = [
            train_eval_tf.mapped_triples,
            valid_eval_tf.mapped_triples,
            test_eval_tf.mapped_triples,
        ]
    metric_results = evaluator.evaluate(
        model=model,
        mapped_triples=split_tf.mapped_triples,
        **kwargs,
    )
    return metric_dict(metric_results)

# 包含了代表性模型 + 对其维度和学习率做了初步调优的配置
model_specs_nations = [
    {
        "name": "TransE",
        "model": "TransE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100, "scoring_fct_norm": 1},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "DistMult",
        "model": "DistMult",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ComplEx",
        "model": "ComplEx",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RotatE",
        "model": "RotatE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 5e-4},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ConvE",
        "model": "ConvE",
        "training_loop": "lcwa",
        "create_inverse": True,  # ConvE 强制要求开启 inverse triples
        "model_kwargs": {
            "embedding_dim": 100,
            "output_channels": 32,
            "input_dropout": 0.2,
            "feature_map_dropout": 0.2,
            "output_dropout": 0.3,
        },
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RESCAL",
        "model": "RESCAL",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
]

kge_nations_results = {}
seed_nations = 42

for spec in model_specs_nations:
    print("\n" + "=" * 90)
    print(f"训练模型: {spec['name']}")
    print("=" * 90)

    train_tf = build_tf_nations(train_triples, create_inverse=spec["create_inverse"])
    valid_tf = build_tf_nations(valid_triples, create_inverse=False)

    start_time = time.time()
    
    # 借助 pykeen 的 pipeline 整合自动带有 Early Stopping
    try:
        result = pipeline(
            training=train_tf,
            validation=valid_tf,
            testing=test_eval_tf,
            model=spec["model"],
            model_kwargs=spec["model_kwargs"],
            training_loop=spec["training_loop"],
            optimizer="adam",
            optimizer_kwargs=spec["optimizer_kwargs"],
            training_kwargs=spec["train_kwargs"],
            stopper="early",
            stopper_kwargs={
                "frequency": 10,
                "patience": 10,  # Nations数据集较小可以稍微等久一点容忍度
                "relative_delta": 0.002,
                "metric": "mean_reciprocal_rank",
            },
            evaluator="RankBasedEvaluator",
            evaluator_kwargs={"filtered": True},
            random_seed=seed_nations,
            device=device_nations,
        )
        elapsed = time.time() - start_time

        model = result.model
        filtered_metrics = evaluate_split_nations(model, test_eval_tf, filtered=True)

        # === 测试 PyKEEN 模型的纯推理速度 ===
        import torch
        hr_batch = torch.tensor([[h, r] for h, r, t in test_triples], device=device_nations)
        model.eval()
        with torch.no_grad():
            # warm-up
            _ = model.score_t(hr_batch[:1])
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            start_infer = time.time()
            _ = model.score_t(hr_batch)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            pure_infer_time = time.time() - start_infer
        
        kge_nations_results[spec["name"]] = {
            "training_time": elapsed,
            "filtered": filtered_metrics,
            "pure_infer_time": pure_infer_time,
        }
        
        print(f"{spec['name']} | filtered MRR={filtered_metrics['MRR']:.4f} | train_time={elapsed:.1f}s | infer_time={pure_infer_time:.4f}s")
        
        # 清理显存避免爆内存
        del model
        del result
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"训练 {spec['name']} 失败: {str(e)}")

print("\n" + "=" * 100)
print("NATIONS: KGE 模型与 NSR 结果汇总对比")
print("=" * 100)

rows = []

# 把我们前面的 Enhanced NSR 结果放进来
if "enhanced_filtered" in globals():
    rows.append({
        "Model": "⭐ Enhanced NSR",
        "MR": enhanced_filtered.get("MR", 0.0),
        "MRR": enhanced_filtered.get("MRR", 0.0),
        "Hits@1": enhanced_filtered.get("Hits@1", 0.0),
        "Hits@3": enhanced_filtered.get("Hits@3", 0.0),
        "Hits@10": enhanced_filtered.get("Hits@10", 0.0),
        "Train Time(s)": f"{enhanced_filt_t:.2f}" if "enhanced_filt_t" in globals() else "-",
        "Infer Time(s)": f"{pure_infer_time:.4f}" if "pure_infer_time" in globals() else "-",
    })

for model_name, results in kge_nations_results.items():
    rows.append({
        "Model": model_name,
        "MR": results["filtered"]["MR"],
        "MRR": results["filtered"]["MRR"],
        "Hits@1": results["filtered"]["Hits@1"],
        "Hits@3": results["filtered"]["Hits@3"],
        "Hits@10": results["filtered"]["Hits@10"],
        "Train Time(s)": f"{results['training_time']:.1f}",
        "Infer Time(s)": f"{results.get('pure_infer_time', 0):.4f}",
    })

df_rows = pd.DataFrame(rows).sort_values("MRR", ascending=False).reset_index(drop=True)
print("\nFiltered Test (Sorted by MRR)")
print("-" * 110)
print(df_rows.to_string(index=False, formatters={
    "MR": lambda x: f"{x:.4f}",
    "MRR": lambda x: f"{x:.4f}",
    "Hits@1": lambda x: f"{x:.2f}%",
    "Hits@3": lambda x: f"{x:.2f}%",
    "Hits@10": lambda x: f"{x:.2f}%",
    "Infer Time(s)": lambda x: f"{x}",
}))

print("\n" + "=" * 100)
print("NATIONS 对比测试完成")
print("=" * 100)


NATIONS: 其他 KGE 模型测试 (使用带有早期停止和针对性超参调优验证)
Device: cuda

训练模型: TransE


Training epochs on cuda:0:   6%|▌         | 9/150 [00:02<00:30,  4.61epoch/s, loss=0.989, prev_loss=1.06]INFO:pykeen.evaluation.evaluator:Evaluation took 0.05s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.3047863841056824. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-d9cd5e06-b7e4-457d-828a-f81fdec01ca2.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:03<00:22,  5.90epoch/s, loss=0.815, prev_loss=0.831]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.3197689354419708. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-d9cd5e06-b7e4-457d-828a-f81fdec01ca2.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  19%|█▉        | 29/150 [00:05<00:18,  6.38epoch/s, los

TransE | filtered MRR=0.3977 | train_time=23.6s | infer_time=0.0002s

训练模型: DistMult


Training epochs on cuda:0:   6%|▌         | 9/150 [00:01<00:16,  8.35epoch/s, loss=0.993, prev_loss=0.994]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.4800083041191101. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-1a9d1ca0-ef8a-4558-b5db-3ed392c0cd89.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:02<00:15,  8.31epoch/s, loss=0.966, prev_loss=0.969]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.5876290798187256. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-1a9d1ca0-ef8a-4558-b5db-3ed392c0cd89.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  19%|█▉        | 29/150 [00:03<00:13,  8.67epoch/s, lo

DistMult | filtered MRR=0.6515 | train_time=17.7s | infer_time=0.0001s

训练模型: ComplEx


...

RotatE | filtered MRR=0.5344 | train_time=27.5s | infer_time=0.0002s

训练模型: ConvE


Training epochs on cuda:0:   0%|          | 0/150 [00:00<?, ?epoch/s]INFO:pykeen.triples.triples_factory:Creating inverse triples.
INFO:pykeen.training.training_loop:Dropping last (incomplete) batch each epoch (1/28 (3.57%) batches).
Training epochs on cuda:0:   6%|▌         | 9/150 [00:02<00:25,  5.63epoch/s, loss=0.332, prev_loss=0.347]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.6452409625053406. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-94078182-5ae1-42e6-9966-6116cc90a771.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:03<00:20,  6.34epoch/s, loss=0.234, prev_loss=0.235]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.6594781279563904. Saved model weights to /home/amax/.data/pykeen/check

ConvE | filtered MRR=0.7900 | train_time=26.0s | infer_time=0.0003s

训练模型: RESCAL


Training epochs on cuda:0:   6%|▌         | 9/150 [00:01<00:17,  8.01epoch/s, loss=17.5, prev_loss=17.6]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.3830898106098175. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-6250494a-ea5e-42c8-a86c-3d71679dca61.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  53%|█████▎    | 79/150 [00:11<00:12,  5.64epoch/s, loss=15.3, prev_loss=15.4]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 80: 0.3879226744174957. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-6250494a-ea5e-42c8-a86c-3d71679dca61.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 80.
Training epochs on cuda:0:  59%|█████▉    | 89/150 [00:13<00:10,  5.63epoch/s, loss=1

RESCAL | filtered MRR=0.5067 | train_time=24.1s | infer_time=0.0002s

NATIONS: KGE 模型与 NSR 结果汇总对比

Filtered Test (Sorted by MRR)
--------------------------------------------------------------------------------------------------------------
   Model     MR    MRR Hits@1 Hits@3 Hits@10 Train Time(s) Infer Time(s)
   ConvE 1.7786 0.7900 66.67% 88.56%  99.50%          26.0        0.0003
DistMult 2.6891 0.6515 50.00% 73.63%  96.52%          17.7        0.0001
  RotatE 3.3881 0.5344 33.58% 66.42%  96.77%          27.5        0.0002
  RESCAL 3.4254 0.5067 29.60% 62.94%  98.26%          24.1        0.0002
 ComplEx 4.0846 0.4333 22.14% 52.74%  96.27%          23.2        0.0002
  TransE 3.3209 0.3977  3.98% 71.39%  98.01%          23.6        0.0002

NATIONS 对比测试完成


## Upgrade NSR On $\texttt{Nations}$
- 注意：需要先run NSR On $\texttt{Nations}$的第一部分**数据集准备**

### Model Definition

In [5]:
def discover_sym_inv_upgraded(train_triples, num_relations, sym_min_support=2, sym_min_conf=0.1, inv_min_support=2, inv_min_conf=0.1):
    """
    升级版：发现对称关系和逆关系，同时返回可用于推理加权的置信度
    """
    pairs_by_r = defaultdict(set)
    pair_to_relations = defaultdict(set)
    
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))
        pair_to_relations[(h, t)].add(r) 

    symmetric_relations = {} # {r: conf}
    for r in range(num_relations):
        pairs = pairs_by_r[r]
        if not pairs: continue
        sym_count = sum(1 for (h, t) in pairs if (t, h) in pairs)
        conf = sym_count / len(pairs)
        if sym_count >= sym_min_support and conf >= sym_min_conf:
            symmetric_relations[r] = conf

    inverse_pairs = {} # {(r1, r2): conf}
    for r1 in range(num_relations):
        p1 = pairs_by_r[r1]
        if not p1: continue
        for r2 in range(num_relations):
            if r1 == r2: continue
            p2 = pairs_by_r[r2]
            if not p2: continue
            inv_count = sum(1 for (h, t) in p1 if (t, h) in p2)
            conf = inv_count / len(p1)
            
            if inv_count >= inv_min_support and conf >= inv_min_conf:
                inverse_pairs[(r1, r2)] = conf
                
    return pair_to_relations, symmetric_relations, inverse_pairs


def discover_multi_topology_upgraded(train_triples, num_relations, min_support=8, min_conf=0.1):
    """
    升级版：同时返回置信度，用于作为权重参与规则融合打分
    """
    pairs_by_r = defaultdict(set)
    for h, r, t in train_triples:
        pairs_by_r[r].add((h, t))
        
    rules_chain, rules_fork1, rules_fork2, rules_rev_chain = {}, {}, {}, {}
    
    for r_i in range(num_relations):
        pi = pairs_by_r[r_i]
        if not pi: continue
        
        h_to_x = defaultdict(list)
        x_to_h = defaultdict(list)
        for h, x in pi:
            h_to_x[h].append(x)
            x_to_h[x].append(h)
            
        for r_j in range(num_relations):
            pj = pairs_by_r[r_j]
            if not pj: continue
            
            x_to_t = defaultdict(list)
            t_to_x = defaultdict(list)
            for x, t in pj:
                x_to_t[x].append(t)
                t_to_x[t].append(x)
                
            chain_pairs, fork1_pairs, fork2_pairs, rev_chain_pairs = set(), set(), set(), set()
            
            for x, heads in x_to_h.items():
                if x in x_to_t:
                    tails = x_to_t[x]
                    for h in heads:
                        for t in tails:
                            chain_pairs.add((h, t))
                            
            for x, heads_i in x_to_h.items():
                if x in t_to_x:
                    heads_j = t_to_x[x]
                    for h in heads_i:
                        for t in heads_j:
                            fork1_pairs.add((h, t))
                            
            for x, tails_i in h_to_x.items():
                if x in x_to_t:
                    tails_j = x_to_t[x]
                    for h in tails_i:
                        for t in tails_j:
                            fork2_pairs.add((h, t))
                            
            for x, tails_i in h_to_x.items():
                if x in t_to_x:
                    heads_j = t_to_x[x]
                    for h in tails_i:
                        for t in heads_j:
                            rev_chain_pairs.add((h, t))
                            
            def eval_rule(pairs, rule_dict):
                sup = len(pairs)
                if sup < min_support: return
                best_rk, best_hit = None, -1
                for r_k in range(num_relations):
                    hit = len(pairs & pairs_by_r[r_k])
                    if hit > best_hit:
                        best_rk, best_hit = r_k, hit
                conf = best_hit / sup
                if conf >= min_conf:
                    rule_dict[(r_i, r_j)] = (best_rk, conf) # 保存最佳目标关系和对应的置信度
                    
            eval_rule(chain_pairs, rules_chain)
            eval_rule(fork1_pairs, rules_fork1)
            eval_rule(fork2_pairs, rules_fork2)
            eval_rule(rev_chain_pairs, rules_rev_chain)
            
    return rules_chain, rules_fork1, rules_fork2, rules_rev_chain


class NSRUpgradedTopology:
    """
    升级版的NSR，引入了置信度权重以及 W_gamma_gamma的混合配置支持可视分析
    """
    def __init__(self, num_entities, num_relations, T=0.5, delta_theta=0.1, Wgg_lv=2, alpha_comp=1, infer_version=1):
        self.N, self.M = num_entities, num_relations
        self.W_beta_beta = np.zeros((self.N * self.M, self.N * self.M))
        self.W_beta_gamma = np.zeros((self.N, self.M))
        
        # 新增 Relation Cluster 层连接
        self.W_gamma_gamma_lv1 = np.zeros((self.M, self.M)) # 基于实例(h,t)严格共现的hebbian权重
        self.W_gamma_gamma_lv2 = np.zeros((self.M, self.M)) # 基于PMI的计算权重
        self.W_gamma_gamma = np.zeros((self.M, self.M))     # 最终关联矩阵
        
        self.T = T # 相近关系过滤阈值
        self.delta_theta = delta_theta # Hebbian增量超参
        self.Wgg_lv = Wgg_lv # 1: 只使用lv1, 2: 只使用lv2
        # 调试期间先默认为1
        self.alpha_comp = alpha_comp # 组合规则的多跳神经传导衰减因子
        self.infer_version = infer_version # 1 为按关联边累加叠加，2 为同关系取置信度最大值
        
        self._tails_by_hr = defaultdict(list)
        self._heads_by_tr = defaultdict(list)
        
        self.symmetric_relations = {} # r_k: conf
        self.inverse_map = defaultdict(list) # r_k: [(r_i, conf)]
        
        self.rules_chain = defaultdict(list) # r_k: [(r_i, r_j, conf)]
        self.rules_fork1 = defaultdict(list) 
        self.rules_fork2 = defaultdict(list) 
        self.rules_rev_chain = defaultdict(list) 

        # 用于统计记录推理来源
        self.stat_total_queries = 0
        self.stat_sym_only = 0
        self.stat_inv_only = 0
        self.stat_mixed = 0
        self.stat_comp_only = 0
        self.stat_base = 0

    def _vec_index(self, e, r): return r * self.N + e

    def fit(self, train_triples):
        self.train_triples = list(train_triples)
        hr_freq = defaultdict(int)
        
        ht_to_relations = defaultdict(set)
        
        for h, r, t in train_triples:
            self.W_beta_beta[self._vec_index(t, r), self._vec_index(h, r)] = 1.0
            self._tails_by_hr[(h, r)].append(t)
            self._heads_by_tr[(t, r)].append(h)
            hr_freq[(h, r)] += 1
            ht_to_relations[(h, t)].add(r) # 严格同头同尾共现
        
        for (h, r), freq in hr_freq.items():
            self.W_beta_gamma[h, r] = 1.0
            
        N_total = len(ht_to_relations)
        C_r = defaultdict(int)
        C_r_r = defaultdict(int)
        
        for ht, relations in ht_to_relations.items():
            for r in relations:
                C_r[r] += 1
            relations_list = list(relations)
            for i in range(len(relations_list)):
                for j in range(i+1, len(relations_list)):
                    r1 = relations_list[i]
                    r2 = relations_list[j]
                    C_r_r[(r1, r2)] += 1
                    C_r_r[(r2, r1)] += 1
                    
        for r_i in range(self.M):
            for r_j in range(self.M):
                if r_i != r_j:
                    if C_r_r[(r_i, r_j)] > 0:
                        self.W_gamma_gamma_lv1[r_i, r_j] = C_r_r[(r_i, r_j)] * self.delta_theta
                        raw_lift = (C_r_r[(r_i, r_j)] * N_total) / (max(1, C_r[r_i]) * max(1, C_r[r_j]))
                        self.W_gamma_gamma_lv2[r_i, r_j] = raw_lift
                    else:
                        self.W_gamma_gamma_lv1[r_i, r_j] = 0.0
                        self.W_gamma_gamma_lv2[r_i, r_j] = 0.0
                        
        max_val = np.max(self.W_gamma_gamma_lv2)
        min_val = np.min(self.W_gamma_gamma_lv2)
        if max_val > min_val:
            self.W_gamma_gamma_lv2 = (self.W_gamma_gamma_lv2 - min_val) / (max_val - min_val)
            
        if self.Wgg_lv == 1:
            max_v1 = np.max(self.W_gamma_gamma_lv1)
            min_v1 = np.min(self.W_gamma_gamma_lv1)
            if max_v1 > min_v1:
                self.W_gamma_gamma = (self.W_gamma_gamma_lv1 - min_v1) / (max_v1 - min_v1)
            else:
                self.W_gamma_gamma = self.W_gamma_gamma_lv1
        else:
            self.W_gamma_gamma = self.W_gamma_gamma_lv2
                        
        return self

    def set_rules(self, sym, inv, chain_r, fork1_r, fork2_r, rev_chain_r):
        self.symmetric_relations = sym
        self.inverse_map.clear()
        self.rules_chain.clear()
        self.rules_fork1.clear()
        self.rules_fork2.clear()
        self.rules_rev_chain.clear()
        
        for (r1, r2), conf in inv.items(): 
            self.inverse_map[r2].append((r1, conf))
        
        for (r_i, r_j), (r_k, conf) in chain_r.items(): self.rules_chain[r_k].append((r_i, r_j, conf))
        for (r_i, r_j), (r_k, conf) in fork1_r.items(): self.rules_fork1[r_k].append((r_i, r_j, conf))
        for (r_i, r_j), (r_k, conf) in fork2_r.items(): self.rules_fork2[r_k].append((r_i, r_j, conf))
        for (r_i, r_j), (r_k, conf) in rev_chain_r.items(): self.rules_rev_chain[r_k].append((r_i, r_j, conf))

    def _compositionality_infer(self, h_q, r_q):
        scores_comp = {}
        for r_i, r_j, conf in self.rules_chain.get(r_q, []):
            for x in self._tails_by_hr.get((h_q, r_i), []):
                amp_hx = self.W_beta_gamma[h_q, r_i]
                for t in self._tails_by_hr.get((x, r_j), []):
                    amp_xt = self.W_beta_gamma[x, r_j]
                    scores_comp[t] = max(scores_comp.get(t, 0.0), amp_hx * amp_xt * conf * self.alpha_comp)

        for r_i, r_j, conf in self.rules_fork1.get(r_q, []):
            for x in self._tails_by_hr.get((h_q, r_i), []):
                amp_hx = self.W_beta_gamma[h_q, r_i]
                for t in self._heads_by_tr.get((x, r_j), []):
                    amp_tx = self.W_beta_gamma[t, r_j]
                    scores_comp[t] = max(scores_comp.get(t, 0.0), amp_hx * amp_tx * conf * self.alpha_comp)

        for r_i, r_j, conf in self.rules_fork2.get(r_q, []):
            for x in self._heads_by_tr.get((h_q, r_i), []):
                amp_xh = self.W_beta_gamma[x, r_i]
                for t in self._tails_by_hr.get((x, r_j), []):
                    amp_xt = self.W_beta_gamma[x, r_j]
                    scores_comp[t] = max(scores_comp.get(t, 0.0), amp_xh * amp_xt * conf * self.alpha_comp)
                    
        for r_i, r_j, conf in self.rules_rev_chain.get(r_q, []):
            for x in self._heads_by_tr.get((h_q, r_i), []):
                amp_xh = self.W_beta_gamma[x, r_i]
                for t in self._heads_by_tr.get((x, r_j), []):
                    amp_tx = self.W_beta_gamma[t, r_j]
                    scores_comp[t] = max(scores_comp.get(t, 0.0), amp_xh * amp_tx * conf * self.alpha_comp)
            
        return scores_comp

    def _single_rel_infer_detailed(self, h_q, r_q):
        scores_sym = {}
        scores_inv = {}
        scores_comp = {}
        
        if r_q in self.symmetric_relations:
            conf = self.symmetric_relations[r_q]
            heads = self._heads_by_tr.get((h_q, r_q), [])
            for x in heads:
                val = float(self.W_beta_gamma[x, r_q]) * conf
                if self.infer_version == 1:
                    scores_sym[x] = scores_sym.get(x, 0.0) + val
                else:
                    scores_sym[x] = max(scores_sym.get(x, 0.0), val)
                
        if r_q in self.inverse_map:
            for inv_r, conf in self.inverse_map[r_q]:
                heads = self._heads_by_tr.get((h_q, inv_r), [])
                for x in heads:
                    val = float(self.W_beta_gamma[x, inv_r]) * conf
                    if self.infer_version == 1:
                        scores_inv[x] = scores_inv.get(x, 0.0) + val
                    else:
                        scores_inv[x] = max(scores_inv.get(x, 0.0), val)
                    
        scores_comp = self._compositionality_infer(h_q, r_q)
        
        merged = {}
        for e in set(scores_sym) | set(scores_inv) | set(scores_comp):
            s_s = scores_sym.get(e, 0.0)
            s_i = scores_inv.get(e, 0.0)
            s_c = scores_comp.get(e, 0.0)
            if self.infer_version == 1:
                merged[e] = s_s + s_i + s_c
            else:
                merged[e] = max(s_s, s_i, s_c)
            
        if not merged:
            col = self.W_beta_beta[:, self._vec_index(h_q, r_q)]
            scores_arr = col.reshape(self.N, self.M, order="F").sum(axis=1)
            base = {int(e): float(s) for e, s in enumerate(scores_arr) if s > 0}
            return base, scores_sym, scores_inv, scores_comp, True
            
        return merged, scores_sym, scores_inv, scores_comp, False

    def infer(self, h_q, r_q):
        self.stat_total_queries += 1
        sim_rels = {r_q: 1.0}
        for r_p in range(self.M):
            if r_p != r_q and self.W_gamma_gamma[r_q, r_p] >= self.T:
                sim_rels[r_p] = self.W_gamma_gamma[r_q, r_p]
                
        final_scores = {}
        any_sym = False
        any_inv = False
        any_comp = False
        any_base = False

        for r_sim, w_gamma in sim_rels.items():
            merged, s_sym, s_inv, s_comp, is_base = self._single_rel_infer_detailed(h_q, r_sim)
            if s_sym: any_sym = True
            if s_inv: any_inv = True
            if s_comp: any_comp = True
            if is_base: any_base = True
            
            for e, s in merged.items():
                val = s * w_gamma
                if self.infer_version == 1:
                    final_scores[e] = final_scores.get(e, 0.0) + val
                else:
                    final_scores[e] = max(final_scores.get(e, 0.0), val)
                
        if any_base and not (any_sym or any_inv or any_comp):
            self.stat_base += 1
        elif any_sym and not any_inv and not any_comp:
            self.stat_sym_only += 1
        elif any_inv and not any_sym and not any_comp:
            self.stat_inv_only += 1
        elif any_comp and not any_sym and not any_inv:
            self.stat_comp_only += 1
        elif any_sym or any_inv or any_comp:
            self.stat_mixed += 1
            
        return final_scores

    def get_inference_stats(self):
        total = self.stat_total_queries
        if total == 0: total = 1
        return {
            "Total Queries": self.stat_total_queries,
            "Base Only": f"{self.stat_base} ({self.stat_base/total*100:.2f}%)",
            "Sym Only": f"{self.stat_sym_only} ({self.stat_sym_only/total*100:.2f}%)",
            "Inv Only": f"{self.stat_inv_only} ({self.stat_inv_only/total*100:.2f}%)",
            "Comp Only": f"{self.stat_comp_only} ({self.stat_comp_only/total*100:.2f}%)",
            "Mixed Rules": f"{self.stat_mixed} ({self.stat_mixed/total*100:.2f}%)",
        }

    def reset_stats(self):
        self.stat_total_queries = 0
        self.stat_sym_only = 0
        self.stat_inv_only = 0
        self.stat_mixed = 0
        self.stat_comp_only = 0
        self.stat_base = 0

    def learn_all_rules(self, min_support=8, min_conf=0.1, sym_min_support=2, sym_min_conf=0.1, inv_min_support=2, inv_min_conf=0.1):
        pair_rels, sym, inv = discover_sym_inv_upgraded(self.train_triples, self.M, sym_min_support, sym_min_conf, inv_min_support, inv_min_conf)
        c, f1, f2, rc = discover_multi_topology_upgraded(self.train_triples, self.M, min_support, min_conf)
        self.set_rules(sym, inv, c, f1, f2, rc)
        
        all_rules = {}
        for k, (rk, conf) in c.items(): all_rules[f"chain_{k}"] = rk
        for k, (rk, conf) in f1.items(): all_rules[f"fork1_{k}"] = rk
        for k, (rk, conf) in f2.items(): all_rules[f"fork2_{k}"] = rk
        for k, (rk, conf) in rc.items(): all_rules[f"rev_chain_{k}"] = rk
        return all_rules, sym, inv
        
    def explain_inference(self, h_q, r_q, id_to_entity, id_to_relation):
        """直观展示某次查询时模型内部具体的计分占比以及各打分项支持的实体"""
        print(f"\n[Explain] Query: ({id_to_entity[h_q]} -- {id_to_relation[r_q]} --> ?)")
        
        sim_rels = {r_q: 1.0}
        for r_p in range(self.M):
            if r_p != r_q and self.W_gamma_gamma[r_q, r_p] >= self.T:
                sim_rels[r_p] = self.W_gamma_gamma[r_q, r_p]
        
        print(f"Triggered Similar Relations (W_gamma_gamma >= {self.T:.2f}):")
        for r, w in sim_rels.items():
            if r != r_q:
                print(f"  - {id_to_relation[r]} (weight: {w:.4f})")
                
        total_sym, total_inv, total_comp = {}, {}, {}
        
        for r_sim, w_gamma in sim_rels.items():
            merged, s_sym, s_inv, s_comp, is_base = self._single_rel_infer_detailed(h_q, r_sim)
            if not is_base:
                for e, s in s_sym.items():
                    val = s * w_gamma
                    if self.infer_version == 1:
                        total_sym[e] = total_sym.get(e, 0.0) + val
                    else:
                        total_sym[e] = max(total_sym.get(e, 0.0), val)
                for e, s in s_inv.items():
                    val = s * w_gamma
                    if self.infer_version == 1:
                        total_inv[e] = total_inv.get(e, 0.0) + val
                    else:
                        total_inv[e] = max(total_inv.get(e, 0.0), val)
                for e, s in s_comp.items():
                    val = s * w_gamma
                    if self.infer_version == 1:
                        total_comp[e] = total_comp.get(e, 0.0) + val
                    else:
                        total_comp[e] = max(total_comp.get(e, 0.0), val)
                
        def get_top1(d):
            if not d: return "None", 0.0
            e_best = max(d.items(), key=lambda x: x[1])
            return id_to_entity[e_best[0]], e_best[1]

        sum_sym = sum(total_sym.values())
        sum_inv = sum(total_inv.values())
        sum_comp = sum(total_comp.values())
        total_score = sum_sym + sum_inv + sum_comp
        
        if total_score > 0:
            print(f"Score Component Percentages & Top Directed Entities:")
            print(f"  Sym:  {sum_sym/total_score*100:6.1f}% -> Top Target: [{get_top1(total_sym)[0]}] (score breakdown: {get_top1(total_sym)[1]:.4f})")
            print(f"  Inv:  {sum_inv/total_score*100:6.1f}% -> Top Target: [{get_top1(total_inv)[0]}] (score breakdown: {get_top1(total_inv)[1]:.4f})")
            print(f"  Comp: {sum_comp/total_score*100:6.1f}% -> Top Target: [{get_top1(total_comp)[0]}] (score breakdown: {get_top1(total_comp)[1]:.4f})")
        else:
            print("  No topology rules triggered. Fully backed-off to base RR inference.")
            
        # 增加: 输出最终得分排名前3的预测实体
        final_scores = self.infer(h_q, r_q)
        sorted_candidates = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
        print("Model Top 3 Predictions:")
        for i, (e, s) in enumerate(sorted_candidates[:3]):
            print(f"  {i+1}. [{id_to_entity[e]}] (Score: {s:.4f})")

### Hyperpara Search（Based On $\texttt{Optuna}$）

关于

A. 默认的 `TPESampler` (Tree-structured Parzen Estimator)

思路：“以史为鉴”。它会记录之前所有 Trial 的参数和结果，把结果分成 “好的组” 和 “差的组”。

动作：它用概率模型（核密度估计）去拟合 “好的参数长什么样”，然后在 “好参数” 的概率密度高的地方采样下一个点。

特点：它是一种基于模型（Model-based）的序贯优化。

B. `CmaEsSampler` (协方差矩阵自适应进化策略)

思路：“物竞天择”。它模拟生物进化，不记录那么多历史，而是维护一个 “正态分布”。

动作：从当前的正态分布中采样出一组 “后代”（即几组超参数）。让这些后代去跑（评估）。

把表现好的后代留下来，更新正态分布的均值（往好的方向挪）和协方差矩阵（决定搜索范围的形状和参数间的关联性）。

特点：它是一种基于进化（Evolutionary）的优化，非常擅长利用参数之间的相关性。

实测效果还是`TPESampler` better


In [6]:
import optuna
import random
import time

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    T_thresh = trial.suggest_float('T_thresh', 0.1, 0.9)
    min_support = trial.suggest_int('min_support', 3, 30)
    min_conf = trial.suggest_float('min_conf', 0.1, 0.8)
    delta_theta = trial.suggest_float('delta_theta', 0.0, 1.0)
    Wgg_lv = trial.suggest_categorical('Wgg_lv', [1, 2])
    # 暂时先设置成0
    alpha_comp = trial.suggest_float('alpha_comp', 0.1,1) # 新增：多跳神经传导衰减因子搜索
    
    model = NSRUpgradedTopology(num_entities, num_relations, T=T_thresh, delta_theta=delta_theta, Wgg_lv=Wgg_lv, alpha_comp=alpha_comp, infer_version=2)
    model.fit(train_triples)
    model.learn_all_rules(
        min_support=min_support, 
        min_conf=min_conf, 
        sym_min_support=2, 
        sym_min_conf=min_conf, 
        inv_min_support=2, 
        inv_min_conf=min_conf
    )
    
    model.reset_stats()
    val_metrics = evaluate_ranking_filtered(lambda h, r: model.infer(h, r), valid_triples, num_entities, all_triples)
    return val_metrics['MRR']

print("Running Optuna study on Validation Set (Filtered Evaluation)...")
study = optuna.create_study(
    direction='maximize',
    # 这是换了一种sampler（注意：sampler需在建立study时传入）
    # sampler=optuna.samplers.CmaEsSampler() 
)
study.optimize(objective, n_trials=100)

print("Best Valid MRR:", study.best_value)
print("Best Params:", study.best_params)

print("\nEvaluating Best Model on Test Set (Filtered Evaluation)...")
best_model = NSRUpgradedTopology(
    num_entities, num_relations, 
    T=study.best_params['T_thresh'], 
    delta_theta=study.best_params['delta_theta'],
    Wgg_lv=study.best_params['Wgg_lv'],
    alpha_comp=study.best_params['alpha_comp'],
    infer_version=2
)
best_model.fit(train_triples)
best_model.learn_all_rules(
    min_support=study.best_params['min_support'],
    min_conf=study.best_params['min_conf'],
    sym_min_support=2,
    sym_min_conf=study.best_params['min_conf'],
    inv_min_support=2,
    inv_min_conf=study.best_params['min_conf']
)

best_model.reset_stats()

# === 正确测试纯推理速度 (不包含评估排序时间) ===
start_time = time.time()
for h, r, t in test_triples:
    _ = best_model.infer(h, r)
pure_infer_time = time.time() - start_time

# 测试完整的评估时间（推理 + 排序/指标计算）
start_time = time.time()
test_metrics = evaluate_ranking_filtered(lambda h, r: best_model.infer(h, r), test_triples, num_entities, all_triples)
total_eval_time = time.time() - start_time

print(f"Test MRR:     {test_metrics['MRR']:.4f}")
print(f"Test Hits@1:  {test_metrics['Hits@1']:.2f}%")
print(f"Test Hits@3:  {test_metrics['Hits@3']:.2f}%")
print(f"Test Hits@10: {test_metrics['Hits@10']:.2f}%")
print(f"Pure Inference Time: {pure_infer_time:.4f}s  (Avg: {pure_infer_time/len(test_triples)*1000:.4f} ms/triple)")
print(f"Total Eval Time:     {total_eval_time:.4f}s  (Avg: {total_eval_time/len(test_triples)*1000:.4f} ms/triple)")

print("\nInference Stats on Test Set:")
for k, v in best_model.get_inference_stats().items():
    print(f"  {k}: {v}")

Running Optuna study on Validation Set (Filtered Evaluation)...
Best Valid MRR: 0.792057509771078
Best Params: {'T_thresh': 0.14871517678634907, 'min_support': 11, 'min_conf': 0.1412878799124261, 'delta_theta': 0.6500015341847969, 'Wgg_lv': 2, 'alpha_comp': 0.27690959488575295}

Evaluating Best Model on Test Set (Filtered Evaluation)...
Test MRR:     0.7993
Test Hits@1:  67.16%
Test Hits@3:  91.04%
Test Hits@10: 99.50%
Pure Inference Time: 0.2610s  (Avg: 1.2984 ms/triple)
Total Eval Time:     0.2574s  (Avg: 1.2806 ms/triple)

Inference Stats on Test Set:
  Total Queries: 402
  Base Only: 0 (0.00%)
  Sym Only: 0 (0.00%)
  Inv Only: 14 (3.48%)
  Comp Only: 0 (0.00%)
  Mixed Rules: 388 (96.52%)


## 训练时间

In [ ]:
import time

print("=== Training NSRUpgradedTopology with Best Params ===")
best_params = study.best_params
# 记录开始时间
start_time = time.time()

# 1. 模型初始化 (直接传入最佳的配置参数)
timed_model = NSRUpgradedTopology(
    num_entities, 
    num_relations, 
    T=best_params['T_thresh'],
    delta_theta=best_params['delta_theta'],
    Wgg_lv=best_params['Wgg_lv'], 
    alpha_comp=best_params['alpha_comp'],
    infer_version=2
)

# 2. 记忆拟合 (构造 W_beta_beta 和 W_gamma_gamma)
timed_model.fit(train_triples)

# 3. 结构与规则发现并加载到模型中 (利用给定的 min_support 和 min_conf)
timed_model.learn_all_rules(
    min_support=best_params['min_support'], 
    min_conf=best_params['min_conf']
)

# 记录结束时间
end_time = time.time()
training_time = end_time - start_time

print(f"训练完成！")
print(f"训练时间 (Training Time): {training_time:.6f} 秒")
print(f"Test MR:     {test_metrics['MR']:.4f}")

## Explainability  On $\texttt{Nations}$

### $W_{\gamma \gamma}$ Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

M = best_model.W_gamma_gamma.shape[0]

# 自动选定一行（找一个大于阈值的项最多、表现最典型的关系行）
# 这里排除了自身到自身的关联，找到激活相关关系最多的一行
row_activations = np.sum((best_model.W_gamma_gamma >= best_model.T) & (np.eye(M) == 0), axis=1)
target_row = np.argmax(row_activations) 
target_rel_name = id_to_relation[target_row]

plt.figure(figsize=(14, 12))
# 不在使用静态 annot_matrix，我们自己灵活掌控 ax.text
ax = sns.heatmap(
    best_model.W_gamma_gamma, 
    cmap="YlGnBu", 
    xticklabels=[id_to_relation[i] for i in range(num_relations)], 
    yticklabels=[id_to_relation[i] for i in range(num_relations)]
)

# 动态打标：主角行显眼大红星，其余行隐蔽小白星
for i in range(M):
    for j in range(M):
        if i != j and best_model.W_gamma_gamma[i, j] >= best_model.T:
            if i == target_row:
                ax.text(j + 0.5, i + 0.5, "★", color="red", size=18, ha="center", va="center", fontweight="bold")
            else:
                ax.text(j + 0.5, i + 0.5, "★", color="blue", size=10, ha="center", va="center", alpha=0.5)

# Annotate T_thresh on the colorbar
colorbar = ax.collections[0].colorbar
cbar_y = best_model.T
colorbar.ax.axhline(cbar_y, color='red', linewidth=3)
colorbar.ax.text(
    1.2, cbar_y, 
    f"T_thresh ({best_model.T:.3f})", 
    va='center', ha='left', color='red', fontweight='bold', fontsize=12
)

plt.title(f"$W_{{\\gamma \\gamma}}$ Heatmap\\n(Activated paths >= {best_model.T:.3f}. Highlighting row: '{target_rel_name}')", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import MDS
from sklearn.cluster import AgglomerativeClustering
import matplotlib.cm as cm
import warnings
warnings.filterwarnings("ignore")

# 1. 转换相似度矩阵 W_gamma_gamma 为距离矩阵
# 距离 = 1.0 - 相似度 (截断以防止由于浮点精度产生微小的负数)
similarity_matrix = best_model.W_gamma_gamma
distance_matrix = np.clip(1.0 - similarity_matrix, 0.0, 1.0)

# 对角线上的距离严格置设为 0 (自己到自己距离为0)
np.fill_diagonal(distance_matrix, 0.0)

# 确保距离矩阵是对称的（MDS要求）
# 虽然 W_gamma_gamma 一般是对称的，但是以防万一
distance_matrix = (distance_matrix + distance_matrix.T) / 2.0

# 2. 降维到 2D 空间 (MDS算法)
# MDS 最适合处理明确的 pairwise distance 矩阵
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42, max_iter=3000)
coords = mds.fit_transform(distance_matrix)

# 3. 对关系进行聚类（以便用不同的颜色标识出“关系小圈子/同配性”）
# 设置聚类数量，比如5或者根据Nations的特性（比如政治、经济、地理、条约等）
num_clusters = 5
clustering = AgglomerativeClustering(n_clusters=num_clusters, metric='precomputed', linkage='average')
labels = clustering.fit_predict(distance_matrix)

# 4. 绘制 2D 散点和标注
plt.figure(figsize=(16, 12))
colors = cm.rainbow(np.linspace(0, 1, num_clusters))

for cluster_id in range(num_clusters):
    idx = np.where(labels == cluster_id)[0]
    plt.scatter(
        coords[idx, 0], coords[idx, 1], 
        color=colors[cluster_id], 
        s=100, 
        label=f"Cluster {cluster_id}"
    )

# 把所有 relation 的名字标在散点旁边
for i in range(num_relations):
    plt.annotate(
        id_to_relation[i], 
        (coords[i, 0], coords[i, 1]),
        xytext=(5, 5), 
        textcoords='offset points',
        fontsize=11,
        color='black',
        alpha=0.8
    )

plt.title("2D Projection of Relations based on $W_{\\gamma\\gamma}$ Distance (MDS Clustering)", fontsize=18, pad=20)
plt.xlabel("MDS Dimension 1", fontsize=14)
plt.ylabel("MDS Dimension 2", fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 对于国家的聚类

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

num_ent = best_model.N
num_rel = best_model.M

# 1. 构建完整的多重邻接矩阵 A (形状: N x N x M)
# A[i, j, r] = 1 表示 国家 i 对 国家 j 存在关系 r
A = np.zeros((num_ent, num_ent, num_rel))
for h in range(num_ent):
    for r in range(num_rel):
        tails = best_model._tails_by_hr.get((h, r), [])
        for t in tails:
            A[h, t, r] = 1.0

# 2. 构建“结构等价性” (Structural Equivalence) 特征向量
# 对于国家 i，它的政治身份由 "它对谁发起了什么(Out)" 和 "谁对它发起了什么(In)" 共同决定
# 我们将这两个视角展平拼接，得到长度为 2 * N * M 的庞大拓扑特征向量
country_features = []
for i in range(num_ent):
    out_behavior = A[i, :, :].flatten() # size: N * M
    in_behavior = A[:, i, :].flatten()  # size: N * M
    country_features.append(np.concatenate([out_behavior, in_behavior]))
    
country_features = np.array(country_features)

# 3. 计算国家间的政治拓扑相似度 (Cosine Similarity)
# 相似度越接近1，说明这两个国家在国际舞台上的外交姿态和受到的待遇越雷同
similarity_matrix = cosine_similarity(country_features)

# 转化为 DataFrame 以便绘图
country_names = [id_to_entity[i] for i in range(num_ent)]
df_sim = pd.DataFrame(similarity_matrix, index=country_names, columns=country_names)

# 4. 可视化与聚类阵营划分
plt.figure(figsize=(10, 8))
# 使用 seaborn 的 clustermap 层级聚类，直接在相似度矩阵上切分出阵营模块
g = sns.clustermap(
    df_sim, 
    cmap='coolwarm', 
    annot=True, 
    fmt=".2f", 
    figsize=(12, 10),
    linewidths=.5,
    tree_kws={'linewidths': 1.5}
)
g.fig.suptitle("NATIONS Political Bloc Discovery via Structural Equivalence", y=1.03, fontsize=16)
plt.show()

# 5. 用 KMeans 量化输出阵营成员
num_blocs = 3
kmeans_c = KMeans(n_clusters=num_blocs, random_state=42)
labels = kmeans_c.fit_predict(country_features)

print("=== 基于多重网络结构等价性的国际阵营划分 ===")
blocs = {i: [] for i in range(num_blocs)}
for ent_id, label in enumerate(labels):
    blocs[label].append(id_to_entity[ent_id])
    
for label, members in blocs.items():
    print(f"阵营 {label}: {members}")

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

print("=========================================================")
print("  回答提问 1: 阵营边缘摇摆国 (Soft Clustering 特性与相似度重合)")
print("=========================================================")
# 计算每个国家到三大阵营的“平均余弦相似度”，寻找脚踏两只船的摇摆国
bloc_centers = {}
for b in range(num_blocs):
    # 提取该阵营所有国家的特征向量并求平均，作为阵营中心
    bloc_indices = [i for i, label in enumerate(labels) if label == b]
    bloc_centers[b] = np.mean(country_features[bloc_indices], axis=0)

for i in range(num_ent):
    ent_name = id_to_entity[i]
    sims = [cosine_similarity([country_features[i]], [bloc_centers[b]])[0][0] for b in range(num_blocs)]
    # 打印每个国家与三大阵营中心的相似度
    print(f"{ent_name:>12} -> 对阵营0: {sims[0]:.3f} | 对阵营1: {sims[1]:.3f} | 对阵营2: {sims[2]:.3f}")

# 顺便检查 commonbloc 相关关系在阵营内部的密度
bloc_rels = [r for r, name in id_to_relation.items() if 'bloc' in name.lower() or 'treaty' in name.lower() or 'alli' in name.lower()]
print(f"\n找到的同盟类关系 ID: {bloc_rels} (名称: {[id_to_relation[r] for r in bloc_rels]})")
for b in range(num_blocs):
    b_nodes = [i for i, label in enumerate(labels) if label == b]
    internal_edges = 0
    possible_edges = len(b_nodes) * (len(b_nodes) - 1) * len(bloc_rels)
    if possible_edges > 0:
        for u in b_nodes:
            for v in b_nodes:
                if u != v:
                    internal_edges += sum([A[u, v, r] for r in bloc_rels])
        print(f"阵营 {b} 内部的同盟关系连边密度: {internal_edges} / {possible_edges} = {internal_edges/possible_edges:.3f}")

print("\n=========================================================")
print("  回答提问 2: 加入 W_beta_gamma 展平拼接后的聚类突变分析")
print("=========================================================")
# W_beta_gamma 是长为 M 的向量，代表国家体量和活跃度偏置
country_features_v2 = []
weight_for_bg = 1.0 # 调节活跃度特征权重的缩放因子

for i in range(num_ent):
    out_behavior = A[i, :, :].flatten()
    in_behavior = A[:, i, :].flatten()
    # 提取并拼接国家在 M 种关系上的整体活跃频次特征
    bg_behavior = best_model.W_beta_gamma[i, :] * weight_for_bg
    country_features_v2.append(np.concatenate([out_behavior, in_behavior, bg_behavior]))

country_features_v2 = np.array(country_features_v2)

# 执行新的聚类
kmeans_v2 = KMeans(n_clusters=3, random_state=42)
labels_v2 = kmeans_v2.fit_predict(country_features_v2)

print("=== 引入 W_beta_gamma (活跃体量参数) 后的新阵营划分 ===")
blocs_v2 = {i: [] for i in range(3)}
for ent_id, label in enumerate(labels_v2):
    blocs_v2[label].append(id_to_entity[ent_id])
    
for label, members in blocs_v2.items():
    print(f"新阵营 {label}: {members}")

#### 人为定义中心

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 建立一个基于重心的 2D 坐标系（等边三角形）
# 三个顶点分别代表三个阵营的核心极点
V = np.array([
    [0, 1],               # 阵营 0: 顶端 (中东/南亚焦点与不结盟阵营)
    [-0.866, -0.5],       # 阵营 1: 左下 (西方体系)
    [0.866, -0.5]         # 阵营 2: 右下 (东方体系)
])

bloc_names = ["Bloc 0 (Non-Aligned/Regional)", "Bloc 1 (Western Bloc)", "Bloc 2 (Eastern Bloc)"]
bloc_colors = ['#2ca02c', '#1f77b4', '#d62728'] # 绿, 蓝, 红

plt.figure(figsize=(10, 9))

# 1. 绘制背景锚点和引力圈
for i in range(3):
    # 绘制极星
    plt.scatter(V[i, 0], V[i, 1], s=500, color=bloc_colors[i], marker='*', edgecolors='black', zorder=3)
    plt.text(V[i, 0], V[i, 1] + 0.12, bloc_names[i], ha='center', fontsize=12, fontweight='bold', color=bloc_colors[i])
    
    # 绘制引力光晕 (代表越靠近中心隶属度越高)
    circle1 = plt.Circle(V[i], 0.3, color=bloc_colors[i], alpha=0.2, zorder=1)
    circle2 = plt.Circle(V[i], 0.6, color=bloc_colors[i], alpha=0.08, zorder=1)
    plt.gca().add_patch(circle1)
    plt.gca().add_patch(circle2)

# 阵营之间画出虚线骨架
triangle = plt.Polygon(V, fill=False, linestyle='--', color='gray', alpha=0.5, zorder=1)
plt.gca().add_patch(triangle)

# 2. 计算映射所有国家在三个引力极内的相对位置
for i in range(num_ent):
    ent_name = id_to_entity[i]
    
    # 提取该国与三个阵营虚拟中心的余弦相似度
    sims = np.array([cosine_similarity([country_features[i]], [bloc_centers[b]])[0][0] for b in range(3)])
    
    # 使用 3次方 来增强差别（因为原始余弦相似度范围集中在 0.3~0.8，三次幂能拉大差距，将核心国吸向顶点）
    weights = sims ** 3
    # 归一化为重心相对权重 (相加等于1)
    weights = weights / np.sum(weights) 
    
    # 根据三点重心定律计算点在二维空间坐标落点
    pos = np.dot(weights, V)
    
    # 获取之前 KMeans 切出的绝对归属标签作为点的底色 (确认分类正确性)
    hard_label = labels[i]
    
    plt.scatter(pos[0], pos[1], s=150, color=bloc_colors[hard_label], edgecolors='white', linewidth=1.5, zorder=4)
    
    # 标注国家名字
    plt.text(pos[0], pos[1] - 0.05, ent_name.upper(), ha='center', fontsize=10, 
             fontweight='bold', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.8), zorder=5)

plt.title("Political Blocs & Swing States (Barycentric Similarity Map)", fontsize=16, y=1.02)
plt.xlim(-1.2, 1.2)
plt.ylim(-0.9, 1.3)
plt.axis('off')
plt.tight_layout()
plt.show()

#### 更真实美观的Version

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 1. 使用 PCA 将高维国家特征映射到自然 2D 空间，从而保留它们真实的空间散布和相对距离
pca = PCA(n_components=2, random_state=42)
country_positions = pca.fit_transform(country_features)

# 2. 在自然 2D 空间中重新进行 KMeans 聚类 (保留基于平面位置寻找聚类中心)
num_blocs = 4  # 您现在可以随意在此处修改需要切分的阵营数目！
kmeans_2d = KMeans(n_clusters=num_blocs, random_state=42)
labels_2d = kmeans_2d.fit_predict(country_positions)
centers_2d = kmeans_2d.cluster_centers_

plt.figure(figsize=(12, 10))

# 动态获取配色以支持任意灵活的聚类数量 (num_blocs)
cmap = plt.get_cmap('tab10' if num_blocs <= 10 else 'tab20')
bloc_colors = [cmap(i) for i in range(num_blocs)]

# 通过聚类标签去映射调色
color_map = [bloc_colors[l] for l in labels_2d]

# 绘制各个国家的散点图位置
for i in range(num_ent):
    ent_name = id_to_entity[i]
    x, y = country_positions[i, 0], country_positions[i, 1]
    
    # 减小marker大小到150
    plt.scatter(x, y, s=120, color=color_map[i], edgecolors='white', linewidth=1.5, zorder=4)
    
    # 标注国家名字 - 使用小写名称匹配，并添加箭头
    if ent_name == 'jordan':
        plt.annotate(ent_name.upper(), 
                    xy=(x, y),  # 箭头指向的位置（marker位置）
                    xytext=(x-0.5, y + 1.5),  # 文字位置（在marker上方）
                    fontsize=12, 
                    fontweight='bold',
                    ha='center',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.8),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1, alpha=0.7),  # 添加箭头
                    zorder=5)
    elif ent_name == 'egypt':
        plt.annotate(ent_name.upper(), 
                    xy=(x, y),  # 箭头指向的位置（marker位置）
                    xytext=(x+0.5, y + 1.5),  # 文字位置（在marker上方）
                    fontsize=12, 
                    fontweight='bold',
                    ha='center',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.8),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1, alpha=0.7),  # 添加箭头
                    zorder=5)
    elif ent_name == 'indonesia':
        plt.annotate(ent_name.upper(), 
                    xy=(x, y),  # 箭头指向的位置（marker位置）
                    xytext=(x + 1.5, y-1.5 ),  # 文字位置（在marker左上方）
                    fontsize=12, 
                    fontweight='bold',
                    ha='center',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.8),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1, alpha=0.7),  # 添加箭头
                    zorder=5)
    elif ent_name == 'burma':
        plt.annotate(ent_name.upper(), 
                    xy=(x, y),  # 箭头指向的位置（marker位置）
                    xytext=(x - 1.5, y - 1.5),  # 文字位置（在marker右上方）
                    fontsize=12, 
                    fontweight='bold',
                    ha='center',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.8),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1, alpha=0.7),  # 添加箭头
                    zorder=5)
    else:
        # 普通文字标注
        plt.text(x, y - 0.5, ent_name.upper(), ha='center', fontsize=12, 
                fontweight='bold', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.8), zorder=5)

# 绘制自然聚类中心 (星号) 及其势力范围 - 减小星号大小到300
for b in range(num_blocs):
    plt.scatter(centers_2d[b, 0], centers_2d[b, 1], 
                s=300, color=bloc_colors[b], marker='*', edgecolors='black', linewidth=1.0, zorder=6, alpha=0.8)
    
    plt.text(centers_2d[b, 0], centers_2d[b, 1] + 0.4, f"Center {b}", 
             ha='center', fontsize=11, fontweight='bold', color=bloc_colors[b], zorder=6)
    
    # 画出同心聚类圈辅以表示归属重力域
    circle1 = plt.Circle(centers_2d[b], 1.5, color=bloc_colors[b], alpha=0.15, zorder=1)
    circle2 = plt.Circle(centers_2d[b], 3.0, color=bloc_colors[b], alpha=0.08, zorder=1)
    plt.gca().add_patch(circle1)
    plt.gca().add_patch(circle2)

# 添加图例 - 使用英文描述
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=bloc_colors[3], markersize=15, label='Center 3 (Red Cluster, Right): Western Capitalist Core'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=bloc_colors[1], markersize=15, label='Center 1 (Orange Cluster, Bottom-Left): Eastern Socialist Core'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=bloc_colors[2], markersize=15, label='Center 2 (Green Cluster, Top-Left): Non-Aligned Third World'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=bloc_colors[0], markersize=15, label='Center 0: Moderate Non-Aligned')
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=12, frameon=True)

plt.title("NATIONS Political Blocs ", fontsize=18, y=1.02)
plt.axhline(0, color='gray', linestyle='--', alpha=0.3, zorder=0)
plt.axvline(0, color='gray', linestyle='--', alpha=0.3, zorder=0)
plt.xlabel("Principal Component 1 ", fontsize=14)
plt.ylabel("Principal Component 2 ", fontsize=14)

# 稍微放大绘图范围以免切除文字
x_min, x_max = np.min(country_positions[:,0]), np.max(country_positions[:,0])
y_min, y_max = np.min(country_positions[:,1]), np.max(country_positions[:,1])
plt.xlim(x_min - 2, x_max + 2)
plt.ylim(y_min - 2, y_max + 2)

plt.tight_layout()
plt.show()


## NSR on $\texttt{KINSHIP}$

理论上不应该在这个Notebook，但是主要是想测试在PyKEEN上的KINSHIP的表现

In [ ]:
# ==================================================================================
# 额外：在 PyKEEN 自带的 Kinship 数据集上测试 Enhanced NSR (带超参数调优)
# ==================================================================================
from pykeen.datasets import Kinships

print("\n" + "=" * 100)
print("Testing Enhanced NSR on PyKEEN: KINSHIP")
print("=" * 100)

# 1. Load PyKEEN Kinship dataset
dataset_kinship = Kinships()

entity_to_id_kin = dataset_kinship.training.entity_to_id
relation_to_id_kin = dataset_kinship.training.relation_to_id
id_to_entity_kin = {v: k for k, v in entity_to_id_kin.items()}
id_to_relation_kin = {v: k for k, v in relation_to_id_kin.items()}

def to_triples_kin(triples_factory):
    triples = triples_factory.mapped_triples.tolist()
    return [(h, r, t) for h, r, t in triples]

train_triples_kin = to_triples_kin(dataset_kinship.training)
valid_triples_kin = to_triples_kin(dataset_kinship.validation)
test_triples_kin = to_triples_kin(dataset_kinship.testing)
all_triples_kin = train_triples_kin + valid_triples_kin + test_triples_kin

num_entities_kin = len(entity_to_id_kin)
num_relations_kin = len(relation_to_id_kin)

print(f"Kinship Entities: {num_entities_kin}")
print(f"Kinship Relations: {num_relations_kin}")
print(f"Kinship Train triples: {len(train_triples_kin)}")
print(f"Kinship Valid triples: {len(valid_triples_kin)}")
print(f"Kinship Test triples: {len(test_triples_kin)}")

# 2 & 3. Build model and Hyperparameter Tuning (Optuna)
import optuna
import time

print("\n" + "-" * 80)
print("Hyperparameter Tuning for Kinship (Optuna)")
print("-" * 80)

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_kin(trial):
    T_thresh = trial.suggest_float('T_thresh', 0.1, 0.9)
    min_support = trial.suggest_int('min_support', 3, 30)
    min_conf = trial.suggest_float('min_conf', 0.1, 0.8)
    delta_theta = trial.suggest_float('delta_theta', 0.0, 1.0)
    Wgg_lv = trial.suggest_categorical('Wgg_lv', [1, 2])
    alpha_comp = trial.suggest_float('alpha_comp', 0.1, 1)
    
    model = NSRUpgradedTopology(
        num_entities_kin, num_relations_kin, 
        T=T_thresh, delta_theta=delta_theta, Wgg_lv=Wgg_lv, alpha_comp=alpha_comp, infer_version=2
    )
    model.fit(train_triples_kin)
    model.learn_all_rules(
        min_support=min_support, 
        min_conf=min_conf, 
        sym_min_support=2, 
        sym_min_conf=min_conf, 
        inv_min_support=2, 
        inv_min_conf=min_conf
    )
    
    model.reset_stats()
    val_metrics = evaluate_ranking_filtered(lambda h, r: model.infer(h, r), valid_triples_kin, num_entities_kin, all_triples_kin)
    return val_metrics['MRR']

print("Running Optuna study on Validation Set (Filtered Evaluation)...")
study_kin = optuna.create_study(direction='maximize')
study_kin.optimize(objective_kin, n_trials=50)

print("Best Valid MRR:", study_kin.best_value)
print("Best Params:", study_kin.best_params)

# 4. Final Evaluation on Test Set
print("\nEvaluating Best Model on Test Set (Filtered Evaluation)...")
best_model_kin = NSRUpgradedTopology(
    num_entities_kin, num_relations_kin, 
    T=study_kin.best_params['T_thresh'], 
    delta_theta=study_kin.best_params['delta_theta'],
    Wgg_lv=study_kin.best_params['Wgg_lv'],
    alpha_comp=study_kin.best_params['alpha_comp'],
    infer_version=2
)
best_model_kin.fit(train_triples_kin)
final_rules_kin, final_sym_kin, final_inv_kin = best_model_kin.learn_all_rules(
    min_support=study_kin.best_params['min_support'],
    min_conf=study_kin.best_params['min_conf'],
    sym_min_support=2,
    sym_min_conf=study_kin.best_params['min_conf'],
    inv_min_support=2,
    inv_min_conf=study_kin.best_params['min_conf']
)

best_model_kin.reset_stats()

start_t_kin = time.perf_counter()
best_test_metrics_kin = evaluate_ranking_filtered(
    lambda h, r: best_model_kin.infer(h, r), 
    test_triples_kin, 
    num_entities_kin, 
    all_triples_kin
)
kin_test_t = time.perf_counter() - start_t_kin

print(f"\nLearned compositional rules count for final config: {len(final_rules_kin)}")
print(f"Symmetric Relations count: {len(final_sym_kin)}")
print(f"Inverse Relations count: {len(final_inv_kin)}\n")

print(f"Best Enhanced NSR on Kinship - Test (filtered) [time={kin_test_t:.2f}s]:")
print(f"Test MRR:     {best_test_metrics_kin['MRR']:.4f}")
print(f"Test Hits@1:  {best_test_metrics_kin['Hits@1']:.2f}%")
print(f"Test Hits@3:  {best_test_metrics_kin['Hits@3']:.2f}%")
print(f"Test Hits@10: {best_test_metrics_kin['Hits@10']:.2f}%")

print("\nInference Stats on Test Set:")
for k, v in best_model_kin.get_inference_stats().items():
    print(f"  {k}: {v}")